
<img src="https://reqlut2.s3.sa-east-1.amazonaws.com/reqlut-images/duoc/logo.png?v=87.8" width="180px"/>


In [1]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))


# Codificación y Preparación de Datos

En esta fase del proyecto se realiza la **transformación de variables** para preparar el dataset para modelos de *Machine Learning*.  
El objetivo es convertir todas las variables en formato **numérico**, reducir la **dimensionalidad** y asegurar que los datos estén listos para el entrenamiento de modelos.

### Objetivos de esta fase

1. Analizar variables categóricas del dataset.
2. Reducir categorías poco frecuentes (especialmente en variables como `Degree`).
3. Aplicar técnicas de codificación:
   - **OneHotEncoder** para variables categóricas nominal.
   - **OrdinalEncoder** para variables categóricas ordinal.
   - **BinaryEncoding** para variables categóricas de alta cardinalidad.
   - **StandardScaler** para variables numéricas.
4. Construir un **pipeline de transformación** usando `ColumnTransformer`.
5. Generar un dataset final listo para modelado.

Dataset utilizado: **Student Depression Dataset (versión limpia)**. Para saber cómo se obtuvo este dataset, consultar el notebook `Fase_2B_limpieza.ipynb`


In [2]:
import numpy as np
import pandas as pd
from src.carga_csv import cargar_csv
from category_encoders import BinaryEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.eda_utils import *

In [3]:
df = cargar_csv(
    r"..\data\processed\Student_Depression_Dataset_Limpio.csv"
)

In [4]:
df_codificado = df.copy()
df_codificado.info()

<class 'pandas.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 15 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27817 non-null  int64  
 1   Gender                                 27817 non-null  str    
 2   Age                                    27817 non-null  float64
 3   City                                   27817 non-null  str    
 4   Academic Pressure                      27817 non-null  float64
 5   CGPA                                   27817 non-null  float64
 6   Study Satisfaction                     27817 non-null  float64
 7   Sleep Duration                         27817 non-null  str    
 8   Dietary Habits                         27817 non-null  str    
 9   Degree                                 27817 non-null  str    
 10  Have you ever had suicidal thoughts ?  27817 non-null  str    
 11  Work/Study Ho

| Método         | Qué hace                                                                                                                         |
| -------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| LabelEncoder   | convierte categorías en números                                                                                                  |
| OneHotEncoder  | crea columnas binarias de categorías nominales                                                                                   |
| OrdinalEncoder | asigna números pero pensado para variables ordinales                                                                             |
| BinaryEncoding | codifica variables categóricas de alta cardinalidad usando representación binaria para reducir la cantidad de columnas generadas |
| StandardScaler | normaliza datos numéricos y los asigna en una misma escala                                                                       |


In [5]:
df_codificado.select_dtypes(include='str').nunique()

Gender                                    2
City                                     30
Sleep Duration                            4
Dietary Habits                            3
Degree                                   28
Have you ever had suicidal thoughts ?     2
Family History of Mental Illness          2
dtype: int64

## Reducción de cardinalidad

Algunas variables categóricas contienen una gran cantidad de categorías (por ejemplo, `Degree`).

Para evitar:
- Explosión de dimensiones en One-Hot Encoding
- Sobreajuste en modelos

Se agrupan las categorías menos frecuentes en una categoría general (por ejemplo: "Other").

In [6]:
# Columnas que tienen muchas categorías y que queremos simplificar
# En este caso se seleccionó 'Degree' porque suele tener muchos valores distintos
cols = ['Degree']

# Recorremos cada columna de la lista
for col in cols:
    # 1. Contamos cuántas veces aparece cada categoría
    # value_counts() devuelve las categorías ordenadas por frecuencia
    # nlargest(10) selecciona solo las 10 más frecuentes
    top = df_codificado[col].value_counts().nlargest(10).index
    
    # 2. Reemplazamos las categorías poco frecuentes
    # Si el valor está dentro de las 10 más comunes se mantiene
    # Si no, se reemplaza por la categoría "Other"
    df_codificado[col] = df_codificado[col].apply(lambda x: x if x in top else 'Other')

In [7]:
df_codificado['Degree'].unique()

<StringArray>
[   'Other',      'BCA',   'M.Tech', 'Class 12',     'B.Ed',      'MSc',
      'BHM',      'MCA',    'B.Com',   'B.Arch',   'B.Tech']
Length: 11, dtype: str


## Construcción del Pipeline de Transformación

Para preparar los datos se utiliza **ColumnTransformer**, lo que permite aplicar diferentes transformaciones según el tipo de variable.

### Transformaciones aplicadas

**Variables categóricas**
- Se utiliza `OneHotEncoder`.
- Se aplica `drop='first'` para eliminar una categoría redundante y evitar multicolinealidad.

**Variable binaria**
- Se utiliza `OrdinalEncoder` para convertir respuestas tipo *Yes/No* en valores numéricos.

**Variables numéricas**
- Se utiliza `StandardScaler` para estandarizar los datos.
- Esto transforma las variables para que tengan:
  - Media = 0
  - Desviación estándar = 1

Esto mejora el rendimiento de varios algoritmos de Machine Learning.


In [8]:
df_codificado.info()

<class 'pandas.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 15 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27817 non-null  int64  
 1   Gender                                 27817 non-null  str    
 2   Age                                    27817 non-null  float64
 3   City                                   27817 non-null  str    
 4   Academic Pressure                      27817 non-null  float64
 5   CGPA                                   27817 non-null  float64
 6   Study Satisfaction                     27817 non-null  float64
 7   Sleep Duration                         27817 non-null  str    
 8   Dietary Habits                         27817 non-null  str    
 9   Degree                                 27817 non-null  str    
 10  Have you ever had suicidal thoughts ?  27817 non-null  str    
 11  Work/Study Ho

In [9]:
cols_ordinal = [
    'Academic Pressure',
    'Study Satisfaction',
    'Financial Stress',
    'Sleep Duration',
    'Dietary Habits'
]

cols_binary = [
    'Degree'
]

In [10]:
for col in cols_ordinal:
    valores = df_codificado[col].unique()

    print(f'\nColumna: {col}')
    print(f'Cantidad de valores únicos: {len(valores)}')
    print(valores)


Columna: Academic Pressure
Cantidad de valores únicos: 6
[5. 2. 3. 4. 1. 0.]

Columna: Study Satisfaction
Cantidad de valores únicos: 6
[2. 5. 3. 4. 1. 0.]

Columna: Financial Stress
Cantidad de valores únicos: 5
[1. 2. 5. 3. 4.]

Columna: Sleep Duration
Cantidad de valores únicos: 4
<StringArray>
['5-6 hours', 'Less than 5 hours', '7-8 hours', 'More than 8 hours']
Length: 4, dtype: str

Columna: Dietary Habits
Cantidad de valores únicos: 3
<StringArray>
['Healthy', 'Moderate', 'Unhealthy']
Length: 3, dtype: str


In [11]:
# Definición explícita del orden de las variables ordinales.
# Este orden será utilizado por OrdinalEncoder para asignar
# valores numéricos respetando la jerarquía natural de cada categoría.

categories_ordinal = [
    # Presión académica: menor nivel -> mayor nivel
    list(sorted(df['Academic Pressure'].unique())),

    # Satisfacción con los estudios: menor satisfacción -> mayor satisfacción
    list(sorted(df['Study Satisfaction'].unique())),

    # Estrés financiero: menor nivel -> mayor nivel
    list(sorted(df['Financial Stress'].unique())),

    # Duración del sueño: menos horas -> más horas
    [
        'Less than 5 hours',
        '5-6 hours',
        '7-8 hours',
        'More than 8 hours'
    ],

    # Hábitos alimenticios: peor calidad -> mejor calidad
    [
        'Unhealthy',
        'Moderate',
        'Healthy'
    ]
]

# Variables categóricas de alta cardinalidad.
# Se utilizará Binary Encoding para reducir la cantidad de columnas
# generadas respecto a One-Hot Encoding, manteniendo información
# relevante de las categorías.

cols_binary = [
    'Degree'   # Carrera o grado académico
]

categories_ordinal

[[np.float64(0.0),
  np.float64(1.0),
  np.float64(2.0),
  np.float64(3.0),
  np.float64(4.0),
  np.float64(5.0)],
 [np.float64(0.0),
  np.float64(1.0),
  np.float64(2.0),
  np.float64(3.0),
  np.float64(4.0),
  np.float64(5.0)],
 [np.float64(1.0),
  np.float64(2.0),
  np.float64(3.0),
  np.float64(4.0),
  np.float64(5.0)],
 ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours'],
 ['Unhealthy', 'Moderate', 'Healthy']]

| Método         | Qué hace                                                                                                                         |
| -------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| LabelEncoder   | convierte categorías en números                                                                                                  |
| OneHotEncoder  | crea columnas binarias de categorías nominales                                                                                   |
| OrdinalEncoder | asigna números pero pensado para variables ordinales                                                                             |
| BinaryEncoding | codifica variables categóricas de alta cardinalidad usando representación binaria para reducir la cantidad de columnas generadas |
| StandardScaler | normaliza datos numéricos y los asigna en una misma escala                                                                       |


In [12]:
ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=categories_ordinal)),
    ('scaler', StandardScaler())
])

# Construcción del Pipeline de Transformación

Para preparar los datos se utiliza un `ColumnTransformer`, el cual permite aplicar distintas transformaciones según el tipo de variable.

## Transformaciones aplicadas

### Variables categóricas nominales
Se utiliza `OneHotEncoder` para transformar variables categóricas sin orden inherente en variables binarias.

- `handle_unknown='ignore'`: evita errores al encontrar categorías no vistas durante el entrenamiento.
- `drop='first'`: elimina una categoría de referencia para evitar redundancia entre variables.

### Variables categóricas de alta cardinalidad
Se utiliza `BinaryEncoder` para variables con una gran cantidad de categorías.

- Reduce la cantidad de columnas generadas respecto a `OneHotEncoder`.
- Mantiene información relevante de las categorías utilizando representación binaria.

### Variables ordinales
Se utiliza un pipeline compuesto por:

1. `OrdinalEncoder`: asigna valores numéricos respetando el orden natural de las categorías.
2. `StandardScaler`: estandariza los valores obtenidos para que tengan media 0 y desviación estándar 1.

### Variables numéricas
Se utiliza `StandardScaler` para estandarizar las variables numéricas continuas.

- Facilita el entrenamiento de modelos sensibles a la escala de los datos.
- Permite que las variables se encuentren en una escala comparable.

### Manejo de columnas restantes
Se utiliza `remainder='drop'` para descartar las columnas que no hayan sido especificadas dentro de los transformadores.


In [13]:
pipeline = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first',
                sparse_output=False
            ),
            [
                'Gender',
                'Have you ever had suicidal thoughts ?',
                'Family History of Mental Illness'
            ]
        ),

        (
            'bin',
            BinaryEncoder(),
            [
                'Degree'
            ]
        ),

        (
            'ord',
            ordinal_pipeline,
            cols_ordinal
        ),

        (
            'num',
            StandardScaler(),
            [
                'Age',
                'CGPA',
                'Work/Study Hours'
            ]
        )
    ],
    remainder='drop'
)

# Separación del Conjunto de Datos

Antes de ajustar el pipeline, se divide el dataset en conjuntos de entrenamiento y prueba.

Esta práctica evita **data leakage**, ya que las estadísticas utilizadas por los transformadores (medias, desviaciones estándar y categorías) se calculan únicamente a partir de los datos de entrenamiento.


In [14]:
# Variables predictoras y variable objetivo
X = df.drop(columns=['Depression'])
y = df['Depression']

# División Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Ajustar el pipeline únicamente con los datos de entrenamiento
X_train_transf = pipeline.fit_transform(X_train)

# Aplicar las mismas transformaciones al conjunto de prueba
X_test_transf = pipeline.transform(X_test)

# Reconstruir DataFrames con nombres de columnas
X_train_transf = pd.DataFrame(
    X_train_transf,
    columns=pipeline.get_feature_names_out(),
    index=X_train.index
)

X_test_transf = pd.DataFrame(
    X_test_transf,
    columns=pipeline.get_feature_names_out(),
    index=X_test.index
)

print("Train:", X_train_transf.shape)
print("Test:", X_test_transf.shape)

Train: (22253, 16)
Test: (5564, 16)


# Reconstrucción de los DataFrames Transformados

Los resultados obtenidos por el pipeline son matrices numéricas.

Para facilitar el análisis y la inspección de los datos transformados, se reconstruyen como DataFrames utilizando los nombres de columnas generados automáticamente por el pipeline.


In [15]:
X_train_transf = pd.DataFrame(
    X_train_transf,
    columns=pipeline.get_feature_names_out(),
    index=X_train.index
)

X_test_transf = pd.DataFrame(
    X_test_transf,
    columns=pipeline.get_feature_names_out(),
    index=X_test.index
)

# Aplicación del Pipeline

El pipeline se ajusta únicamente utilizando el conjunto de entrenamiento.

Posteriormente, las mismas transformaciones son aplicadas al conjunto de prueba, garantizando que ambos conjuntos sean procesados de manera consistente.

In [16]:
# Crear datasets finales incorporando la variable objetivo
train_df = X_train_transf.copy()
train_df['Depression'] = y_train.values

test_df = X_test_transf.copy()
test_df['Depression'] = y_test.values

# Exportar datasets procesados
train_df.to_csv(
    r'..\data\processed\Student_Depression_Dataset_Train.csv',
    index=False
)

test_df.to_csv(
    r'..\data\processed\Student_Depression_Dataset_Test.csv',
    index=False
)

# Exportación del dataset codificado completo

In [17]:
# Variables predictoras
X = df.drop(columns=['Depression'])

# Aplicar todas las transformaciones
X_transf = pipeline.fit_transform(X)

# Reconstruir DataFrame con los nombres de columnas generados
df_transf = pd.DataFrame(
    X_transf,
    columns=pipeline.get_feature_names_out(),
    index=df.index
)

# Incorporar nuevamente la variable objetivo
df_transf['Depression'] = df['Depression'].values

# Visualizar resultado
df_transf.head()

,cat__Gender_Male,cat__Have you ever had suicidal thoughts ?_Yes,cat__Family History of Mental Illness_Yes,bin__Degree_0,bin__Degree_1,bin__Degree_2,bin__Degree_3,bin__Degree_4,ord__Academic Pressure,ord__Study Satisfaction,ord__Financial Stress,ord__Sleep Duration,ord__Dietary Habits,num__Age,num__CGPA,num__Work/Study Hours,Depression
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.345273,-0.694638,-1.489063,-0.354328,1.374420,1.476593,0.895400,-1.122130,1
1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.827109,1.510988,-0.793223,-0.354328,0.119629,-0.370683,-1.200717,-1.122130,0
2,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,-0.102982,1.510988,-1.489063,-1.241729,1.374420,1.066087,-0.429182,0.496561,0
3,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.102982,-0.694638,1.294297,0.533072,0.119629,0.450328,-1.412377,-0.852348,1
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.621145,0.040570,-1.489063,-0.354328,0.119629,-0.165430,0.321870,-1.661693,0


In [18]:
df_transf.to_csv(
    r'..\data\processed\Student_Depression_Dataset_Codificado.csv',
    index=False
)

## Verificación de resultados

Se verifica que:

- Las transformaciones hayan sido aplicadas correctamente.
- Los nombres de las columnas generadas coincidan con las transformaciones realizadas.
- Los conjuntos de entrenamiento y prueba mantengan la estructura esperada.
- La variable objetivo haya sido incorporada correctamente en ambos datasets.

In [19]:
print(train_df.shape)
print(test_df.shape)

train_df.head()

(22253, 17)
(5564, 17)


,cat__Gender_Male,cat__Have you ever had suicidal thoughts ?_Yes,cat__Family History of Mental Illness_Yes,bin__Degree_0,bin__Degree_1,bin__Degree_2,bin__Degree_3,bin__Degree_4,ord__Academic Pressure,ord__Study Satisfaction,ord__Financial Stress,ord__Sleep Duration,ord__Dietary Habits,num__Age,num__CGPA,num__Work/Study Hours,Depression
10640,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.342450,-0.692243,-0.090078,-1.239966,0.121880,0.242172,-0.358151,0.496245,1
6084,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.618804,1.514176,-0.785601,-0.353132,-1.133187,0.242172,0.664384,-0.043712,1
16774,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.342450,-1.427716,1.300969,-1.239966,1.376948,-0.167751,1.080214,0.496245,1
27077,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.104841,-0.692243,-1.481125,-0.353132,1.376948,-1.602482,1.516496,-0.853649,0
25242,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,-0.104841,0.778703,0.605446,-0.353132,0.121880,0.242172,-1.435221,-0.313691,1
